# Week 12 — Execution and Reinforcement Learning

**Focus:** How do actions meet market impact, execution cost, and an environment that responds?

*Live-coding notebook*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/smc77/uc_finmlai/blob/main/lectures/week12/week12_demos.ipynb)

## What this notebook is about

Execution is where a proposed action meets prices, depth, fills, delay, and cost. It is also where reinforcement learning becomes fundamentally different from prediction: the action can change what happens next.

The demonstrations walk a limit-order book, challenge an impact model, use a one-state bandit to expose exploration cost, inspect an action-responsive execution contract, solve one stochastic gridworld exactly, and compare that solution with a frozen tabular Q-learning policy and its visitation ledger.

By the end, you should be able to:

- turn an order into a transparent fill and shortfall ledger;
- distinguish a fixed replay from an action-responsive environment;
- treat a multi-armed bandit as one-state reinforcement learning and measure regret;
- define state, action, reward, transition, and termination;
- compare a learned policy with a known or simple policy baseline; and
- separate training episodes from independent evaluation paths.

The simulated environments are deliberately small enough to inspect. Success inside them is evidence about the declared simulator, not automatically about live execution.

### Course context

For a concise review and the optional textbook route, see [Week 12 summary](https://github.com/smc77/uc_finmlai/blob/main/lectures/week12/week12_summary.md) and [Week 12 reading guide](https://github.com/smc77/uc_finmlai/blob/main/lectures/week12/reading.md). The notebook itself is designed to remain understandable without them.

## How to use this notebook

At the first break, run any **Setup** cell and then the **Imports** cell. If there is no Setup cell, begin with Imports. After that, use the slide cue to jump to the named demo; you do not need to rerun the whole notebook at every break. To check the whole notebook from a clean start, use **Runtime → Run all**.

Each demo follows the same rhythm: predict what the output should show, run the cell, and follow the task immediately underneath it. An **After you run** cell is editable: double-click it, replace `[write here]`, and press Shift-Enter. When an editable research-record entry appears, its completed comparison sits below it in a collapsed box—write your version before opening that box. This notebook is generated from the same source code the lecturer runs live.

## Jump to a demonstration

Both links jump within *this* notebook — neither opens a new tab. Use **Colab** when the notebook is open in Colab, **Jupyter** when it is open in Jupyter or rendered to HTML. You do not need to scroll through or rerun the whole notebook.

> If the **Jupyter** links do nothing in JupyterLab or Notebook 7, that is the windowed renderer, not a broken link: cells outside the viewport are not in the page, so there is no anchor to scroll to. Set *Settings → Settings Editor → Notebook → Windowing mode* to `defer` (or `none`) and they work.

### Deck A

<ul>
<li>Demo 0 — Walk a declared ask book — <a href="#Demo-0-%E2%80%94-Walk-a-declared-ask-book">Jupyter</a> · <a href="#scrollTo=md-8965369c3e35">Colab</a></li>
<li>Demo 1 — Challenge an impact law — <a href="#Demo-1-%E2%80%94-Challenge-an-impact-law">Jupyter</a> · <a href="#scrollTo=md-4e1e265ea91e">Colab</a></li>
</ul>

### Deck B

<ul>
<li>Guided experiment — Explore a one-state bandit — <a href="#Guided-experiment-%E2%80%94-Explore-a-one-state-bandit">Jupyter</a> · <a href="#scrollTo=md-aaff6358c081">Colab</a></li>
<li>Demo 2 — Inspect an execution-environment contract — <a href="#Demo-2-%E2%80%94-Inspect-an-execution-environment-contract">Jupyter</a> · <a href="#scrollTo=md-69bcc56db5cc">Colab</a></li>
<li>Demo 3 — Change one reward in the exact gridworld — <a href="#Demo-3-%E2%80%94-Change-one-reward-in-the-exact-gridworld">Jupyter</a> · <a href="#scrollTo=md-b7aa9330a734">Colab</a></li>
<li>Demo 4 — Train and freeze one Q-learning policy — <a href="#Demo-4-%E2%80%94-Train-and-freeze-one-Q-learning-policy">Jupyter</a> · <a href="#scrollTo=md-cdcd2bc3c648">Colab</a></li>
<li>Demo 5 — Audit frozen value and visitation across training budgets — <a href="#Demo-5-%E2%80%94-Audit-frozen-value-and-visitation-across-training-budgets">Jupyter</a> · <a href="#scrollTo=md-53c2cb2daaaa">Colab</a></li>
</ul>

## Project research-record checkpoint

Update the single project-root `RESEARCH_RECORD.md`; do not duplicate it here. If the project crosses into execution or reinforcement learning, add the applicable fields:

1. **Execution contract:** decision and arrival times, eligible orders, spread, depth or impact, fees, fill and non-fill rules.
2. **Reconciliation:** paper value, fills, slippage, fees, completion, and net implementation shortfall.
3. **Environment:** bandit or sequential setting; state or context, action, reward, transition or simulator version, constraints, and termination.
4. **Learning evidence:** behavior-policy support, baseline policy, independent paths, seeds, and frozen evaluation rule.
5. **Stress:** changed liquidity, costs, dynamics, or simulator assumptions and the resulting failure bound.

A fixed price replay is not an action-responsive environment. Log any design change and whether it occurred before or after the test boundary was opened.

### Imports

> **Run this once before any demo.** It loads the packages and helper functions used below; there is no result to interpret in this cell.

In [ ]:
import io
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle


def _find_repo_root():
    """Locate the repository when available; return None in a standalone Colab."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    if "__file__" in globals():
        source = Path(__file__).resolve()
        candidates.extend([source.parent, *source.parents])
    return next((p for p in candidates if (p / "lectures" / "week12").exists()), None)


REPO_ROOT = _find_repo_root()
LECTURE_FIGURES = None if REPO_ROOT is None else REPO_ROOT / "lectures" / "week12" / "figures"


def save_lecture_figure(filename):
    """Save beside the lecture figures without touching an identical artifact."""
    if LECTURE_FIGURES is None:
        return
    LECTURE_FIGURES.mkdir(parents=True, exist_ok=True)
    output = LECTURE_FIGURES / filename
    buffer = io.BytesIO()
    plt.savefig(buffer, format="png", dpi=180, bbox_inches="tight")
    payload = buffer.getvalue()
    if not output.exists() or output.read_bytes() != payload:
        output.write_bytes(payload)


def finish_plot():
    """Show figures interactively and close them in noninteractive checks."""
    if "agg" not in plt.get_backend().lower():
        plt.show()
    plt.close()

### Demo 0 — Walk a declared ask book

> **Break cue:** Deck A, after recording segment S1. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
ask_price = np.array([100.005, 100.015, 100.025, 100.035, 100.045, 100.055])
ask_size = np.array([180, 260, 420, 650, 900, 1200])
midpoint = 100.00
half_spread_bps = (ask_price[0] / midpoint - 1) * 10_000

print("Declared ask book:")
print(" shares   marginal   average   half-spread   depth   total")
for quantity in (100, 300, 700, 2200):
    remaining = quantity
    cash = 0.0
    marginal = np.nan
    for price, depth in zip(ask_price, ask_size):
        filled = min(remaining, depth)
        cash += filled * price
        remaining -= filled
        if filled:
            marginal = price
        if remaining == 0:
            break
    average = cash / quantity
    concession_bps = (average / midpoint - 1) * 10_000
    depth_bps = concession_bps - half_spread_bps
    print(
        f" {quantity:>6}   {marginal:>8.3f}   {average:>7.4f}"
        f"      {half_spread_bps:>5.2f}      {depth_bps:>5.2f}   {concession_bps:>5.2f} bps"
    )

### Mini-project: walk the book

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 0 — Walk a declared ask book**.

Using the displayed ladder or `week12_demos.ipynb`:

1. price market buys of four sizes;
2. separate half-spread from depth concession;
3. calculate the marginal and average price;
4. state what the snapshot cannot reveal about a passive fill.

**Research record:** the four order sizes; half-spread separated from depth concession; marginal and average price for each; the four-field fill-assumption note and what the snapshot cannot reveal.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 1 — Challenge an impact law

> **Break cue:** Deck A, after recording segment S2. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
g = np.random.default_rng(1201)
participation = np.linspace(0.001, 0.35, 180)
true_impact = 0.0015 * np.sqrt(participation)
observed_impact = true_impact + g.standard_normal(len(participation)) * 0.00004
fit_mask = participation <= 0.08
X_linear = participation[:, None]
X_sqrt = np.sqrt(participation)[:, None]


def fit_one_feature(x, y, mask):
    design = np.column_stack([np.ones(mask.sum()), x[mask, 0]])
    return np.linalg.lstsq(design, y[mask], rcond=None)[0]


coef_linear = fit_one_feature(X_linear, observed_impact, fit_mask)
coef_sqrt = fit_one_feature(X_sqrt, observed_impact, fit_mask)
pred_linear = coef_linear[0] + coef_linear[1] * participation
pred_sqrt = coef_sqrt[0] + coef_sqrt[1] * np.sqrt(participation)
test_mask = participation > 0.08
rmse_linear = np.sqrt(np.mean((pred_linear[test_mask] - observed_impact[test_mask]) ** 2))
rmse_sqrt = np.sqrt(np.mean((pred_sqrt[test_mask] - observed_impact[test_mask]) ** 2))
print(
    f"Held-out high-participation RMSE: linear={rmse_linear * 1e4:.2f} bp, "
    f"square-root={rmse_sqrt * 1e4:.2f} bp"
)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.scatter(participation[fit_mask], observed_impact[fit_mask] * 1e4, s=10, color="0.6", label="fit region")
ax.scatter(participation[test_mask], observed_impact[test_mask] * 1e4, s=10, color="0.82", label="held-out region")
ax.plot(participation, pred_linear * 1e4, label="linear fit")
ax.plot(participation, pred_sqrt * 1e4, label="square-root fit")
ax.axvline(0.08, color="k", ls=":", lw=1)
ax.set_xlabel("participation rate")
ax.set_ylabel("impact (bp)")
ax.set_title("Two laws fit locally; extrapolation separates them")
ax.legend()
save_lecture_figure("demo1_impact_extrapolation.png")
finish_plot()

### Mini-project: challenge the impact law

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 1 — Challenge an impact law**.

Fit linear and square-root impact on a low-participation region.

1. freeze both fits;
2. extrapolate to larger orders;
3. compare held-out errors;
4. identify where the two frozen predictions begin to separate materially.

**Research record:** the low-participation fit region; both frozen laws; held-out error for each; where their predictions separate; one bounded extrapolation claim.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Guided experiment — Explore a one-state bandit

> **Demonstration:** State what you expect before running the cell; afterward, explain which output supports or contradicts that expectation.

In [ ]:
bandit_means = np.array([0.00, 0.04, 0.06])
bandit_noise = 0.12
bandit_horizon = 300
bandit_replications = 500
bandit_epsilons = (0.0, 0.1, 0.3)
bandit_results = {}

for epsilon in bandit_epsilons:
    rng = np.random.default_rng(12020)
    counts = np.zeros((bandit_replications, len(bandit_means)), dtype=int)
    estimates = np.zeros_like(counts, dtype=float)
    cumulative_regret = np.zeros(bandit_replications)
    regret_paths = np.zeros((bandit_replications, bandit_horizon))
    rows = np.arange(bandit_replications)

    for round_number in range(bandit_horizon):
        if round_number < len(bandit_means):
            actions = np.full(bandit_replications, round_number, dtype=int)
        else:
            tie_breaker = rng.random(estimates.shape) * 1e-12
            greedy_actions = np.argmax(estimates + tie_breaker, axis=1)
            explore = rng.random(bandit_replications) < epsilon
            random_actions = rng.integers(0, len(bandit_means), bandit_replications)
            actions = np.where(explore, random_actions, greedy_actions)

        rewards = rng.normal(bandit_means[actions], bandit_noise)
        counts[rows, actions] += 1
        estimates[rows, actions] += (
            rewards - estimates[rows, actions]
        ) / counts[rows, actions]
        cumulative_regret += bandit_means.max() - bandit_means[actions]
        regret_paths[:, round_number] = cumulative_regret

    bandit_results[epsilon] = regret_paths

print("One-state bandit: cumulative pseudo-regret after 300 rounds")
print(" epsilon   mean    p10    p90")
for epsilon, paths in bandit_results.items():
    final = paths[:, -1]
    p10, p90 = np.quantile(final, [0.1, 0.9])
    print(f"   {epsilon:>3.1f}     {final.mean():>4.2f}   {p10:>4.2f}   {p90:>4.2f}")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
colors = {0.0: "#9e2f2f", 0.1: "#2e7d32", 0.3: "#1f5a94"}
rounds = np.arange(1, bandit_horizon + 1)
for epsilon, paths in bandit_results.items():
    mean_path = paths.mean(axis=0)
    p10, p90 = np.quantile(paths, [0.1, 0.9], axis=0)
    ax.plot(rounds, mean_path, color=colors[epsilon], lw=2.2, label=fr"$\varepsilon={epsilon:.1f}$")
    ax.fill_between(rounds, p10, p90, color=colors[epsilon], alpha=0.10)
ax.set_xlabel("decision round")
ax.set_ylabel("cumulative pseudo-regret")
ax.set_title("A one-state problem still charges for exploration")
ax.legend(frameon=False, ncol=3)
save_lecture_figure("guided_bandit_regret.png")
finish_plot()

### Demo 2 — Inspect an execution-environment contract

> **Break cue:** Deck B, after recording segment S4. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
EXECUTION_CONTRACT = {
    "observation": "time, shares remaining, midprice, spread, displayed ask depth",
    "actions": "wait, buy 100, buy 250; infeasible sizes are masked",
    "response": "fills cross spread and depth cost; each trade moves the next midprice",
    "reward": "negative execution, holding, and terminal non-completion costs (dollars)",
    "initial/reset": "600-share buy; fixed prices; seeded market shocks",
    "horizon/terminal": "four decisions; remaining shares cost $0.05 each",
}

print("Declared execution-environment contract:")
for field, definition in EXECUTION_CONTRACT.items():
    print(f"  {field:>16}: {definition}")


class TinyExecutionEnv:
    """A small action-responsive fixture, not a calibrated market model."""

    action_sizes = np.array([0, 100, 250])

    def __init__(self, horizon=4):
        self.horizon = horizon

    def reset(self, seed=1202):
        self.rng = np.random.default_rng(seed)
        self.t = 0
        self.remaining = 600
        self.decision_price = 100.000
        self.arrival_price = 100.012
        self.mid = self.arrival_price
        self.spread = 0.010
        self.ask_depth = 300
        self.fills = []
        return self.observation()

    def observation(self):
        return {
            "time": self.t,
            "remaining": self.remaining,
            "mid": self.mid,
            "spread": self.spread,
            "ask_depth": self.ask_depth,
        }

    def action_mask(self):
        mask = self.action_sizes <= self.remaining
        if self.t == self.horizon - 1 and self.remaining > 0:
            mask[0] = False
        return mask

    def step(self, action):
        mask = self.action_mask()
        if action < 0 or action >= len(mask) or not mask[action]:
            raise ValueError(f"action {action} is infeasible under mask {mask.tolist()}")
        shares = int(self.action_sizes[action])
        execution_cost = 0.0
        fill_price = np.nan
        if shares:
            depth_fraction = shares / self.ask_depth
            depth_cost = 0.012 * depth_fraction**2
            fill_price = self.mid + self.spread / 2 + depth_cost
            execution_cost = shares * (fill_price - self.arrival_price)
            self.fills.append((shares, fill_price))
            self.remaining -= shares
        holding_cost = 0.001 * self.remaining
        self.mid += self.rng.normal(0.0, 0.003) + 0.002 * (shares / self.ask_depth)
        self.t += 1
        done = self.remaining == 0 or self.t == self.horizon
        terminal_cost = 0.05 * self.remaining if done and self.remaining else 0.0
        reward = -(execution_cost + holding_cost + terminal_cost)
        return self.observation(), reward, done, {
            "shares": shares,
            "fill_price": fill_price,
            "execution_cost": execution_cost,
            "holding_cost": holding_cost,
            "terminal_cost": terminal_cost,
        }


env = TinyExecutionEnv()
env.reset(seed=1202)
trace = []
for action in (1, 2, 2):
    mask_before = env.action_mask().tolist()
    observation, reward, done, info = env.step(action)
    trace.append((env.t, action, mask_before, observation["remaining"], reward, info))
    if done:
        break

print("\nAction-responsive trace (actions 1, 2, 2):")
print(" step  action  mask before       fill      remaining    reward")
for step_number, action, mask, remaining, reward, info in trace:
    fill = "--" if np.isnan(info["fill_price"]) else f"{info['fill_price']:.4f}"
    print(f"   {step_number}      {action}    {str(mask):<18} {fill:>8}       {remaining:>4}     {reward:>7.2f}")

filled_shares = sum(shares for shares, _ in env.fills)
execution_vwap = sum(shares * price for shares, price in env.fills) / filled_shares
delay_cost = filled_shares * (env.arrival_price - env.decision_price)
execution_cost = filled_shares * (execution_vwap - env.arrival_price)
total_shortfall = filled_shares * (execution_vwap - env.decision_price)
print("\nImplementation-shortfall reconciliation:")
print(f"  delay cost:           ${delay_cost:,.2f}")
print(f"  execution cost:       ${execution_cost:,.2f}")
print(f"  total shortfall:      ${total_shortfall:,.2f}")
print(f"  reconciliation error: ${total_shortfall - delay_cost - execution_cost:.2e}")

### Mini-project: write the environment contract

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 2 — Inspect an execution-environment contract**.

Inspect the declared fixture and its three-action trace:

1. list the observation fields and feasible actions;
2. explain one state-dependent action mask;
3. identify how an order changes the next state;
4. reconcile the printed shortfall ledger;
5. name one economically relevant hidden state.

**Research record:** observation; actions and masks; action-responsive transition; reward components and units; reset, horizon, terminal rule; shortfall reconciliation; one suspected hidden state.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 3 — Change one reward in the exact gridworld

> **Break cue:** Deck B, after recording segment S5. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
# This is the same stochastic 5x6 gridworld used by the lecture figures.
ROWS, COLS = 5, 6
START, GOAL, PIT = (4, 0), (0, 5), (3, 4)
WALLS = {(1, 1), (1, 2), (2, 2), (3, 2)}
ACTIONS = np.array([[-1, 0], [1, 0], [0, -1], [0, 1]])
ACTION_NAMES = np.array(["up", "down", "left", "right"])
ARROWS = np.array(["↑", "↓", "←", "→"])
PERPENDICULAR = {0: (2, 3), 1: (2, 3), 2: (0, 1), 3: (0, 1)}
STATES = [(r, c) for r in range(ROWS) for c in range(COLS) if (r, c) not in WALLS]
NONTERMINAL = [state for state in STATES if state not in {GOAL, PIT}]
STATE_INDEX = {state: index for index, state in enumerate(NONTERMINAL)}


def move(state, action):
    row = state[0] + ACTIONS[action, 0]
    col = state[1] + ACTIONS[action, 1]
    candidate = (row, col)
    if row < 0 or row >= ROWS or col < 0 or col >= COLS or candidate in WALLS:
        return state
    return candidate


def outcomes(state, action, goal_reward=3.0, pit_reward=-4.0, step_reward=-0.05):
    if state in {GOAL, PIT}:
        return [(1.0, state, 0.0, True)]
    choices = [(0.8, action), (0.1, PERPENDICULAR[action][0]), (0.1, PERPENDICULAR[action][1])]
    merged = {}
    for probability, actual_action in choices:
        successor = move(state, actual_action)
        done = successor in {GOAL, PIT}
        reward = goal_reward if successor == GOAL else pit_reward if successor == PIT else step_reward
        key = (successor, reward, done)
        merged[key] = merged.get(key, 0.0) + probability
    return [(probability, *key) for key, probability in merged.items()]


def q_from_v(values, gamma=0.95, **reward):
    q_values = np.zeros((ROWS, COLS, 4))
    for state in NONTERMINAL:
        for action in range(4):
            q_values[state][action] = sum(
                probability * (payoff + (0.0 if done else gamma * values[successor]))
                for probability, successor, payoff, done in outcomes(state, action, **reward)
            )
    return q_values


def value_iteration(gamma=0.95, tolerance=1e-11, **reward):
    values = np.zeros((ROWS, COLS))
    residuals = []
    start_values = []
    for _ in range(10000):
        q_values = q_from_v(values, gamma, **reward)
        updated = values.copy()
        for state in NONTERMINAL:
            updated[state] = q_values[state].max()
        residual = max(abs(updated[state] - values[state]) for state in NONTERMINAL)
        residuals.append(residual)
        start_values.append(updated[START])
        values = updated
        if residual < tolerance:
            break
    return values, q_from_v(values, gamma, **reward), np.array(residuals), np.array(start_values)


def evaluate_policy(policy, gamma=0.95, **reward):
    transition = np.zeros((len(NONTERMINAL), len(NONTERMINAL)))
    expected_reward = np.zeros(len(NONTERMINAL))
    for state in NONTERMINAL:
        row = STATE_INDEX[state]
        for probability, successor, payoff, done in outcomes(state, int(policy[state]), **reward):
            expected_reward[row] += probability * payoff
            if not done:
                transition[row, STATE_INDEX[successor]] += probability
    solved = np.linalg.solve(np.eye(len(NONTERMINAL)) - gamma * transition, expected_reward)
    return {state: solved[index] for state, index in STATE_INDEX.items()}


def policy_panel(axis, values, policy, title):
    visible = np.full((ROWS, COLS), np.nan)
    for state in NONTERMINAL:
        visible[state] = values[state]
    image = axis.imshow(visible, cmap="RdYlGn", vmin=np.nanmin(visible), vmax=np.nanmax(visible))
    for row in range(ROWS):
        for col in range(COLS):
            state = (row, col)
            if state in WALLS:
                axis.add_patch(Rectangle((col - 0.5, row - 0.5), 1, 1, facecolor="#3f3f3f"))
            elif state == GOAL:
                axis.text(col, row, "GOAL", ha="center", va="center", fontweight="bold", color="#3f7754")
            elif state == PIT:
                axis.text(col, row, "PIT", ha="center", va="center", fontweight="bold", color="#9f3a2f")
            else:
                axis.text(col, row - 0.08, ARROWS[int(policy[state])], ha="center", va="center", fontsize=16)
                axis.text(col, row + 0.31, f"{values[state]:.2f}", ha="center", va="center", fontsize=7)
    axis.scatter([START[1]], [START[0]], s=150, facecolors="none", edgecolors="#285a84", lw=2)
    axis.set(xticks=[], yticks=[], title=title)
    return image


V_STAR, Q_STAR, RESIDUALS, START_VALUES = value_iteration()
POLICY_STAR = np.argmax(Q_STAR, axis=2)
V_COSTLY, Q_COSTLY, _, _ = value_iteration(step_reward=-0.20)
POLICY_COSTLY = np.argmax(Q_COSTLY, axis=2)
changed_states = [state for state in NONTERMINAL if POLICY_STAR[state] != POLICY_COSTLY[state]]

print(f"Baseline step reward -0.05: start value {V_STAR[START]:.4f}")
print(f"Changed step reward -0.20:  start value {V_COSTLY[START]:.4f}")
print(f"States whose greedy action changes: {len(changed_states)}")
for state in changed_states:
    print(f"  {state}: {ACTION_NAMES[POLICY_STAR[state]]} -> {ACTION_NAMES[POLICY_COSTLY[state]]}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4))
policy_panel(axes[0], V_STAR, POLICY_STAR, "Baseline: step reward −0.05")
policy_panel(axes[1], V_COSTLY, POLICY_COSTLY, "Intervention: step reward −0.20")
fig.suptitle("One reward clause changes values and selected actions", y=1.02)
fig.tight_layout()
save_lecture_figure("demo3_reward_intervention.png")
finish_plot()

### Mini-project: change one reward

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demo 3 — Change one reward in the exact gridworld**.

Using `week12_demos.ipynb`:

1. compute the exact policy;
2. change only the step reward from $-0.05$ to $-0.20$;
3. recompute the policy and start value;
4. identify every cell whose action changes.

**Research record:** the exact policy before the change; the single reward clause altered; the recomputed policy and start value; every cell whose action changed.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

### Demo 4 — Train and freeze one Q-learning policy

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
def sample_step(random_generator, state, action, **reward):
    possible = outcomes(state, action, **reward)
    selected = random_generator.choice(len(possible), p=[item[0] for item in possible])
    _, successor, payoff, done = possible[selected]
    return successor, payoff, done


def q_learning(seed, budgets=(25, 50, 100, 250, 500, 1000, 2500), gamma=0.95, epsilon=0.2):
    random_generator = np.random.default_rng(seed)
    q_values = np.zeros((ROWS, COLS, 4))
    visits = np.zeros_like(q_values, dtype=int)
    records = []
    budget_set = set(budgets)
    for episode in range(1, max(budgets) + 1):
        state = START
        for _ in range(200):
            action = int(random_generator.integers(4)) if random_generator.random() < epsilon else int(np.argmax(q_values[state]))
            successor, payoff, done = sample_step(random_generator, state, action)
            visits[state][action] += 1
            learning_rate = visits[state][action] ** (-0.6)
            target = payoff if done else payoff + gamma * np.max(q_values[successor])
            q_values[state][action] += learning_rate * (target - q_values[state][action])
            state = successor
            if done:
                break
        if episode in budget_set:
            frozen_policy = np.argmax(q_values, axis=2)
            frozen_value = evaluate_policy(frozen_policy, gamma)
            coverage = np.mean([visits[s][a] >= 10 for s in NONTERMINAL for a in range(4)])
            agreement = np.mean([frozen_policy[s] == POLICY_STAR[s] for s in NONTERMINAL])
            state_regrets = np.maximum(
                0.0,
                np.array([V_STAR[state] - frozen_value[state] for state in NONTERMINAL]),
            )
            records.append(
                (
                    episode,
                    frozen_value[START],
                    V_STAR[START] - frozen_value[START],
                    coverage,
                    agreement,
                    np.mean(state_regrets),
                    np.max(state_regrets),
                )
            )
    return q_values, visits, np.array(records)


Q_LEARNED, VISITS, LEARNING_RECORDS = q_learning(seed=3307)
LEARNED_POLICY = np.argmax(Q_LEARNED, axis=2)
(
    final_episode,
    final_value,
    final_regret,
    final_coverage,
    final_agreement,
    final_mean_all_state_regret,
    final_worst_all_state_regret,
) = LEARNING_RECORDS[-1]

print("Frozen one-seed Q-learning audit:")
print(f"  training episodes:                          {int(final_episode)}")
print(f"  exact optimal start value:                  {V_STAR[START]:.4f}")
print(f"  frozen learned-policy start value:          {final_value:.4f}")
print(f"  exact start-state regret:                   {final_regret:.4f}")
print(f"  mean regret across accessible start states: {final_mean_all_state_regret:.4f}")
print(f"  worst regret across accessible start states: {final_worst_all_state_regret:.4f}")
print(f"  state-action pairs with at least 10 visits: {final_coverage:.1%}")
print(f"  agreement with one exact greedy policy:     {final_agreement:.1%}")

print("\nFrozen learned policy (S=start, G=goal, P=pit, #=wall):")
for row in range(ROWS):
    labels = []
    for col in range(COLS):
        state = (row, col)
        if state in WALLS:
            labels.append("#")
        elif state == START:
            labels.append("S")
        elif state == GOAL:
            labels.append("G")
        elif state == PIT:
            labels.append("P")
        else:
            labels.append(ARROWS[LEARNED_POLICY[state]])
    print(" ".join(labels))

### Demo 5 — Audit frozen value and visitation across training budgets

> **Break cue:** Deck B, after recording segment S6. Read the task on the preceding slide, predict what the output should show, then run the cell. Record what the task asks for: the specific values you observed, the sample and settings that produced them, and one conclusion the result cannot support.

In [ ]:
episodes = LEARNING_RECORDS[:, 0]
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.8))
axes[0].plot(episodes, LEARNING_RECORDS[:, 1], marker="o", color="#285a84", label="frozen policy")
axes[0].axhline(V_STAR[START], color="black", ls=":", label="exact optimum")
axes[0].set_xscale("log")
axes[0].set(xlabel="training episodes", ylabel="exact frozen-policy value at start", title="Frozen value approaches the answer key")
axes[0].legend(fontsize=8)
axes[1].plot(episodes, LEARNING_RECORDS[:, 3], marker="o", color="#9f3a2f")
axes[1].set_xscale("log")
axes[1].set_ylim(0, 1)
axes[1].set(xlabel="training episodes", ylabel="fraction of pairs with at least 10 visits", title="Coverage remains a separate question")
fig.tight_layout()
save_lecture_figure("demo5_learning_audit.png")
finish_plot()

print("One seed illustrates the calculation; the forty-seed lecture figure shows run-to-run variation.")

### Mini-project: audit one learned policy

> **Editable lecture task.** Complete this cell after running the named demo. The prompt mirrors the lecture mini-project even when the required work is a written classification, audit, diagram, or decision rather than additional code.

**Notebook:** run **Demos 4 and 5 — Train, freeze, and audit one Q-learning policy**.

1. compare the exact and frozen-policy start values;
2. report start-state regret, then mean and worst regret across every accessible start;
3. report policy agreement and ten-visit coverage;
4. read the value and coverage panels across training budgets;
5. distinguish this one-seed illustration from the forty-seed lecture result.

**Research record:** seed and training budget; exact and frozen start values; start, mean-all-start, and worst-all-start regret; policy agreement; ten-visit coverage; the one-seed boundary; one simulator assumption to stress.

#### Your response

Replace the placeholders below with the requested output before opening the lecture debrief.

- **Result or artifact:** `[write here]`
- **Research-record fields:** `[write here]`
- **Bounded conclusion:** `[write here]`

## Where this leaves us

A decision policy must be evaluated inside an environment that responds to its actions, with fills, costs, rewards, and state transitions reconciled explicitly. A learned policy can master a simulator while exploiting assumptions that do not survive outside it.

Week 13 asks what policy evidence is available from logged actions and how causal methods identify the effect of an intervention.